# Génération Prooftag réelle — diffusion et SRPG en direct

Ce notebook **ne lit pas une archive**. Il charge Stable Diffusion 1.5 et ControlNet sur la RTX du serveur, génère une nouvelle image, exécute les 40 pas DDIM/SRPG et affiche chaque prédiction `x0` avec sa carte d'erreurs. Le navigateur est sur le PC, mais le kernel et le GPU sont sur `pcIA`.

## 0. Paramètres de l'expérience
Modifier ici le payload, le prompt et les paramètres. Conserver une seed fixe pour comparer deux réglages.

In [ ]:
PAYLOAD = "https://example.prooftag.test/t/notebook-live-001"
PROMPT = (
    "elegant botanical packaging illustration, organic leaves and flowers, "
    "premium print design, high detail"
)
NEGATIVE_PROMPT = "text, watermark, blurry, low quality, plain QR code, barcode"
SEED = 42
ERROR_CORRECTION = "H"
BASE_STEPS = 12
BASE_STRENGTH = 0.90
GUIDANCE_SCALE = 12.0
BASE_CONTROLNET_SCALE = 1.50
SRPG_STEPS = 40
SRPG_STRENGTH = 1.00  # réglage E005a actuel; modifier seulement pour une ablation nommée
SRPG_CONTROLNET_SCALE = 1.35
SRPG_QR_WEIGHT = 500.0
SRPG_PERCEPTUAL_WEIGHT = 3.0
DISPLAY_EVERY = 1  # 1 = voir chaque pas; 5 = affichage plus léger

## 1. Vérification du kernel GPU et préparation du dossier de résultats

In [ ]:
import csv
import json
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Markdown, display
from PIL import Image, ImageOps

from prooftag_qr.backends import ControlNetBackend
from prooftag_qr.config import Settings
from prooftag_qr.qr import generate_qr, module_error_rate
from prooftag_qr.quality import image_change_metrics
from prooftag_qr.schemas import GenerationRequest
from prooftag_qr.srpg import SRPGConfig, run_srpg_controlnet_img2img
from prooftag_qr.validation import QRValidator

if not torch.cuda.is_available():
    raise RuntimeError(
        "Ce notebook doit utiliser le kernel du serveur RTX, pas Python local sur le PC."
    )
run_name = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + f"-seed-{SEED}"
run_dir = Path("/data/notebook-runs") / run_name
run_dir.mkdir(parents=True, exist_ok=False)
display(Markdown(f"**GPU :** `{torch.cuda.get_device_name(0)}`  \n**Résultats :** `{run_dir}`"))

## 2. Chargement réel de Stable Diffusion et ControlNet
Le premier chargement peut prendre plusieurs minutes. Les poids sont lus depuis le même PVC de cache que l'API.

In [ ]:
settings = Settings(
    data_dir=Path("/data"),
    model_cache_dir=Path("/cache"),
    default_backend="controlnet",
    controlnet_pipeline_mode="img2img",
    device="cuda",
    srpg_enabled=False,
    guided_rediffusion_enabled=False,
    latent_refinement_enabled=False,
)
request = GenerationRequest(
    payload=PAYLOAD,
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    backend="controlnet",
    error_correction=ERROR_CORRECTION,
    seed=SEED,
    steps=BASE_STEPS,
    strength=BASE_STRENGTH,
    guidance_scale=GUIDANCE_SCALE,
    controlnet_scale=BASE_CONTROLNET_SCALE,
    max_attempts=1,
)
backend = ControlNetBackend(settings)
pipeline = backend._load()
display(
    Markdown(f"Pipeline chargée en `{pipeline.unet.dtype}` sur `{pipeline._execution_device}`.")
)

## 3. Génération du QR technique qui guidera le modèle

In [ ]:
blueprint = generate_qr(PAYLOAD, ERROR_CORRECTION, size=512)
blueprint.image.save(run_dir / "00_qr_control.png")
matrix_size = blueprint.matrix.shape[0]
display(
    Markdown(
        f"QR version **{blueprint.version}**, matrice **{matrix_size}×{matrix_size} modules**."
    )
)
display(blueprint.image)

## 4. Première diffusion : génération artistique brute
Cette cellule appelle réellement SD 1.5 + ControlNet. L'image obtenue devient la référence artistique.

In [ ]:
torch.cuda.reset_peak_memory_stats()
raw = backend.generate(request, blueprint, SEED)
raw.save(run_dir / "01_raw_diffusion.png")
raw_error = module_error_rate(raw, blueprint)
peak_cuda_mib = torch.cuda.max_memory_allocated() / 1024**2
display(Markdown(f"**Erreur module brute : {raw_error:.3%}** — pic CUDA : {peak_cuda_mib:.0f} MiB"))
display(raw)

## 5. Lecture de la sortie brute avant toute correction

In [ ]:
validator = QRValidator()


def validate_image(image):
    records = validator.validate(image, PAYLOAD)
    exact = sum(record.exact_payload_match for record in records)
    return records, exact / len(records) if records else 0.0


raw_records, raw_scan_rate = validate_image(raw)
raw_successes = sum(record.exact_payload_match for record in raw_records)
display(Markdown(f"**Lecture brute : {raw_successes}/{len(raw_records)} = {raw_scan_rate:.1%}**"))

## 6. Deuxième diffusion : les 40 pas DDIM/SRPG en direct
À chaque pas, le panneau gauche est la prédiction propre `x0` vue par les losses. À droite, le rouge indique les modules dont le centre est encore faux. Les images sont calculées maintenant et sauvegardées dans `data/notebook-runs/`.

In [ ]:
history = []
status_handle = display(HTML("<b>Initialisation SRPG…</b>"), display_id=True)
image_handle = display(Image.new("RGB", (1024, 512), "white"), display_id=True)


def show_srpg_step(preview, step):
    history.append(step)
    error_rgb = ImageOps.colorize(preview.active_module_map, black="black", white="red").convert(
        "RGB"
    )
    panel = Image.new("RGB", (1024, 512), "white")
    panel.paste(preview.predicted_clean_image.resize((512, 512)), (0, 0))
    panel.paste(error_rgb.resize((512, 512)), (512, 0))
    preview.predicted_clean_image.save(run_dir / f"02_srpg_step_{step.index:02d}_x0.png")
    preview.active_module_map.save(run_dir / f"02_srpg_step_{step.index:02d}_errors.png")
    status_handle.update(
        HTML(
            f"<b>Pas {step.index + 1}/{SRPG_STEPS}</b> — timestep {step.timestep} — "
            f"erreur centrale {step.module_error_rate:.3%} — SRL {step.scanning_robust_loss:.4f} — "
            f"LPIPS {step.perceptual_loss:.4f} — Δ bruit {step.noise_delta_rms:.4f}"
        )
    )
    image_handle.update(panel)


srpg_generator = torch.Generator(device=settings.device).manual_seed(
    (SEED + settings.srpg_seed_offset) % (2**32)
)
srpg = run_srpg_controlnet_img2img(
    pipeline,
    raw,
    blueprint,
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    generator=srpg_generator,
    config=SRPGConfig(
        steps=SRPG_STEPS,
        strength=SRPG_STRENGTH,
        controlnet_scale=SRPG_CONTROLNET_SCALE,
        qr_weight=SRPG_QR_WEIGHT,
        perceptual_weight=SRPG_PERCEPTUAL_WEIGHT,
        functional_weight=4.0,
        max_noise_delta_rms=2.0,
        max_mean_absolute_change=0.20,
        min_relative_module_improvement=0.10,
        save_step_previews=True,
        preview_interval=DISPLAY_EVERY,
    ),
    preview_callback=show_srpg_step,
)
srpg.image.save(run_dir / "03_srpg_unrepaired.png")
gate_reason = srpg.rejection_reason or "accepté"
display(
    Markdown(
        f"**SRPG terminé :** erreur réelle {srpg.initial_module_error_rate:.3%} → "
        f"{srpg.final_module_error_rate:.3%}; MAE {srpg.mean_absolute_change:.4f}; "
        f"porte interne `{srpg.accepted}` ({gate_reason})."
    )
)

## 7. Courbes réellement produites pendant cette exécution

In [ ]:
steps = list(srpg.steps)
x = [step.index for step in steps]
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
series = [
    ("module_error_rate", "Erreur centrale"),
    ("scanning_robust_loss", "Scanning Robust Loss"),
    ("perceptual_loss", "LPIPS"),
    ("noise_delta_rms", "Delta bruit RMS"),
]
for axis, (field, title) in zip(axes.flat, series, strict=True):
    axis.plot(x, [getattr(step, field) for step in steps], marker="o", markersize=3)
    axis.set_title(title)
    axis.grid(alpha=0.25)
    axis.set_xlabel("Pas DDIM")
plt.tight_layout()
fig.savefig(run_dir / "04_srpg_curves.png", dpi=140)

## 8. Validation indépendante de la sortie SRPG
C'est cette mesure, et non la loss interne, qui dit si l'image se scanne réellement.

In [ ]:
srpg_records, srpg_scan_rate = validate_image(srpg.image)
srpg_successes = sum(record.exact_payload_match for record in srpg_records)
display(
    Markdown(
        f"**Lecture SRPG sans réparation : {srpg_successes}/{len(srpg_records)} = "
        f"{srpg_scan_rate:.1%}**"
    )
)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for axis, title, image in zip(axes, ["Brut", "SRPG non réparé"], [raw, srpg.image], strict=True):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()

## 9. Réparations candidates, validation de chacune et sélection finale
Cette étape montre exactement où les modules visibles sont ajoutés. Toutes les variantes sont testées avec les deux décodeurs et les treize dégradations.

In [ ]:
repair_settings = settings.model_copy(
    update={
        "srpg_enabled": False,
        "guided_rediffusion_enabled": False,
        "latent_refinement_enabled": False,
    }
)
repair_backend = ControlNetBackend(repair_settings)
evaluated = []


def evaluate_repair_chain(source, prefix):
    for name, image in repair_backend.variants(source, blueprint, request=request, seed=SEED):
        full_name = f"{prefix}{name}"
        records, pass_rate = validate_image(image)
        change = image_change_metrics(image, raw)
        row = {
            "name": full_name,
            "image": image,
            "pass_rate": pass_rate,
            "module_error_rate": module_error_rate(image, blueprint),
            **change,
        }
        evaluated.append(row)
        image.save(run_dir / f"05_variant_{full_name}.png")
        print(
            "{:32s} scan={:6.1%} module={:6.2%} MAE={:.4f}".format(
                full_name, pass_rate, row["module_error_rate"], row["mean_absolute_change"]
            )
        )


evaluate_repair_chain(srpg.image, "srpg_")
evaluate_repair_chain(raw, "raw_")
for row in evaluated:
    # La production n'autorise une variante SRPG que si la sortie SRPG
    # a elle-même passé son contrôle.
    row["eligible"] = not row["name"].startswith("srpg_") or srpg.accepted
accepted = [
    row
    for row in evaluated
    if row["eligible"] and row["pass_rate"] >= settings.validation_min_pass_rate
]
if accepted:
    selected = min(
        accepted, key=lambda row: (row["mean_absolute_change"], row["changed_pixel_ratio"])
    )
else:
    eligible = [row for row in evaluated if row["eligible"]]
    selected = min(
        eligible,
        key=lambda row: (-row["pass_rate"], row["module_error_rate"], row["mean_absolute_change"]),
    )
final = selected["image"]
final.save(run_dir / "06_final_selected.png")
display(
    Markdown(
        "**Sélection : `{}` — scan {:.1%} — MAE {:.4f}.**".format(
            selected["name"], selected["pass_rate"], selected["mean_absolute_change"]
        )
    )
)

## 10. Comparaison finale et export reproductible

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for axis, title, image in zip(
    axes,
    ["Contrôle QR", "Diffusion brute", "SRPG", f"Final: {selected['name']}"],
    [blueprint.image, raw, srpg.image, final],
    strict=True,
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
fig.savefig(run_dir / "07_comparison.png", dpi=140)
manifest = {
    "run_name": run_name,
    "payload": PAYLOAD,
    "prompt": PROMPT,
    "seed": SEED,
    "parameters": {
        "base_steps": BASE_STEPS,
        "base_strength": BASE_STRENGTH,
        "guidance_scale": GUIDANCE_SCALE,
        "base_controlnet_scale": BASE_CONTROLNET_SCALE,
        "srpg_steps": SRPG_STEPS,
        "srpg_strength": SRPG_STRENGTH,
        "srpg_controlnet_scale": SRPG_CONTROLNET_SCALE,
        "srpg_qr_weight": SRPG_QR_WEIGHT,
        "srpg_perceptual_weight": SRPG_PERCEPTUAL_WEIGHT,
    },
    "raw_scan_rate": raw_scan_rate,
    "srpg_scan_rate": srpg_scan_rate,
    "selected_variant": selected["name"],
    "final_scan_rate": selected["pass_rate"],
    "srpg_internal_accepted": srpg.accepted,
    "srpg_rejection_reason": srpg.rejection_reason,
}
(run_dir / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
with (run_dir / "variants.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(
        stream,
        fieldnames=[
            "name",
            "eligible",
            "pass_rate",
            "module_error_rate",
            "changed_pixel_ratio",
            "mean_absolute_change",
        ],
    )
    writer.writeheader()
    writer.writerows({key: row[key] for key in writer.fieldnames} for row in evaluated)
result_browser_path = f"results/notebook-runs/{run_name}"
display(
    Markdown(f"Résultats persistants disponibles dans **`{result_browser_path}`** dans Jupyter.")
)